In [ ]:
!pip -q install transformers accelerate sentencepiece unidecode tqdm editdistance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.0 MB/s eta 0:00:00


In [ ]:
%%writefile qa_polka_compat.py
import re
import random
from typing import List, Tuple, Optional, Dict

import torch
from torch.nn import functional as F
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
)
from unidecode import unidecode
from tqdm import tqdm

RNG_SEED = 123
random.seed(RNG_SEED)

DEF_Q_PATH = "task4_questions.txt"
DEF_A_PATH = "task4_answers.txt"

OUT_FOUND = "found_answers.txt"
OUT_CORR  = "correct_answers.txt"

def log_probs_from_logits(logits, labels):
    logp = F.log_softmax(logits, dim=-1)
    logp_label = torch.gather(logp, 2, labels.unsqueeze(2)).squeeze(-1)
    return logp_label

@torch.no_grad()
def conditional_logprob(tokenizer, model, prefix, continuation, device="cpu"):
    text_full = prefix + continuation
    full = tokenizer(text_full, return_tensors='pt').to(device)
    out = model(**full)
    logits = out.logits[:, :-1, :]
    ids    = full["input_ids"][:, 1:]
    pref_ids = tokenizer(prefix, return_tensors='pt').to(device)["input_ids"]
    pref_len = pref_ids.shape[1]
    logp_tok = log_probs_from_logits(logits, ids)
    start = max(pref_len - 1, 0)
    cand_logp = logp_tok[:, start:].sum().item()
    return cand_logp

NUM_WORDS_PL = {
    "zero": 0, "jeden": 1, "jedna":1, "jedno":1, "pierwszy":1, "raz":1,
    "dwa": 2, "dwie": 2, "drugi":2,
    "trzy": 3, "trzecia":3, "trzeci":3,
    "cztery": 4, "czwarty":4,
    "piec":5, "piaty":5, "pięć":5, "piąty":5,
    "szesc":6, "szosty":6, "sześć":6, "szósty":6,
    "siedem":7, "siodmy":7, "siódmy":7,
    "osiem":8, "osmy":8, "ósmy":8,
    "dziewiec":9, "dziewiaty":9, "dziewięć":9, "dziewiąty":9,
    "dziesiec":10, "dziesiaty":10, "dziesięć":10, "dziesiąty":10,
    "jedenascie":11, "jedenaście":11,
}

def norm_text(s: str) -> str:
    s = s.strip()
    s = s.replace("„","").replace("”","").replace('"','').replace("’","'").replace("–","-")
    s = re.sub(r"\s+", " ", s)
    s = s.lower()
    s = unidecode(s)
    return s

def answers_from_line(line: str) -> List[str]:
    parts = [p.strip() for p in re.split(r"\t+", line.strip()) if p.strip()]
    return parts if parts else [""]

def is_yesno(q: str) -> bool:
    return norm_text(q).startswith("czy ")

def is_binary_choice(q: str) -> bool:
    return " czy " in norm_text(q) and not is_yesno(q)

MATH_PATTERNS = [
    re.compile(r"w ciagu (\d+)\s*godz\w*\s+przejechalismy\s+(\d+)\s*km", re.I),
    re.compile(r"jesli\s+poruszamy\s+sie\s+(\d+)\s*km/h,\s*to\s*w\s*ile\s*minut\s*pokonamy\s*(\d+)\s*km", re.I),
    re.compile(r"ile\s+wynosi\s+wartosc\s+bezwzgledna\s+liczby\s*-\s*(\d+)", re.I),
    re.compile(r"ile\s+metrow\s+kwadratowych\s+zawiera\s+kilometr\s+kwadratowy", re.I),
    re.compile(r"ile\s+litrow\s+ma\s+jeden\s+decymetr\s+szescienny", re.I),
    re.compile(r"ile\s+kwintali\s+ma\s+tona", re.I),
    re.compile(r"ile\s*stopni\s*wynosi\s*temperatura\s*wrzenia\s*wody\s*w\s*skali\s*fahrenheita", re.I),
    re.compile(r"ile\s+wynosi\s+pierwiastek\s+trzeciego\s+stopnia\s+z\s+osmiu", re.I),
    re.compile(r"ile\s+wspolrzednych\s+trzeba\s+podac.*na plaszczyznie", re.I),
]

def is_math_like(q: str) -> bool:
    qn = unidecode(q.lower())
    return any(p.search(qn) for p in MATH_PATTERNS)

def solve_math(q: str) -> Optional[str]:
    qn = unidecode(q.lower())
    m = re.search(r"w ciagu (\d+)\s*godz\w*\s+przejechalismy\s+(\d+)\s*km", qn)
    if m:
        t = int(m.group(1)); d = int(m.group(2))
        v = d / t
        return f"{int(v)} km/h" if v.is_integer() else f"{v} km/h"
    m = re.search(r"jesli\s+poruszamy\s+sie\s+(\d+)\s*km/h,\s*to\s*w\s*ile\s*minut\s*pokonamy\s*(\d+)\s*km", qn)
    if m:
        v = int(m.group(1)); d = int(m.group(2))
        minutes = int(round((d / v) * 60))
        return str(minutes)
    m = re.search(r"ile\s+wynosi\s+wartosc\s+bezwzgledna\s+liczby\s*-\s*(\d+)", qn)
    if m:
        return m.group(1)
    if re.search(r"ile\s+metrow\s+kwadratowych\s+zawiera\s+kilometr\s+kwadratowy", qn):
        return "1000000"
    if re.search(r"ile\s+litrow\s+ma\s+jeden\s+decymetr\s+szescienny", qn):
        return "1"
    if re.search(r"ile\s+kwintali\s+ma\s+tona", qn):
        return "10"
    if re.search(r"ile\s*stopni\s*wynosi\s*temperatura\s*wrzenia\s*wody\s*w\s*skali\s*fahrenheita", qn):
        return "212"
    if re.search(r"ile\s+wynosi\s+pierwiastek\s+trzeciego\s+stopnia\s+z\s+osmiu", qn):
        return "2"
    if re.search(r"ile\s+wspolrzednych\s+trzeba\s+podac.*na plaszczyznie", qn):
        return "2"
    return None

def load_generation_model():
    use_cuda = torch.cuda.is_available()
    device_id = 0 if use_cuda else -1
    mid = "eryk-mazus/polka-1.1b"
    tok = AutoTokenizer.from_pretrained(mid, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        mid,
        torch_dtype=torch.float16 if use_cuda else torch.float32
    )
    if use_cuda:
        model = model.to("cuda")
    gen = pipeline("text-generation", model=model, tokenizer=tok, device=device_id)
    print(f"[OK] generacja: {mid} @ {'cuda' if use_cuda else 'cpu'}")
    return {"tok": tok, "model": model, "gen": gen}

def load_probability_model():
    mid = "flax-community/papuGaPT2"
    tok = AutoTokenizer.from_pretrained(mid)
    model = AutoModelForCausalLM.from_pretrained(mid)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"[OK] prawdopodobieństwa: {mid} @ {device}")
    return {"tok": tok, "model": model, "device": device}

SYS_PREFIX = (
    "Odpowiadaj krotko i rzeczowo po polsku. "
    "Jesli pytanie zawiera dwie opcje, wybierz jedna z nich. "
    "Jesli pytanie wymaga jednego slowa/liczby, podaj samo slowo/liczbe.\n"
)

def build_fewshot(train_examples: List[Tuple[str, str]], k: int = 6) -> str:
    if not train_examples:
        return ""
    ex = random.sample(train_examples, min(k, len(train_examples)))
    return "\n".join([f"Pytanie: {q}\nOdpowiedz: {a}" for q,a in ex]) + "\n"

def gen_answer(genb, q: str, fewshot_ctx: Optional[str]) -> str:
    prompt = SYS_PREFIX + (fewshot_ctx or "") + f"Pytanie: {q}\nOdpowiedz:"
    out = genb["gen"](
        prompt, max_new_tokens=32, do_sample=True, temperature=0.6, top_p=0.9,
        pad_token_id=genb["tok"].eos_token_id
    )[0]["generated_text"]
    tail = out.split("Odpowiedz:")[-1]
    tail = tail.strip().split("\n")[0].strip()
    tail = re.split(r"[\.!?]", tail)[0].strip()
    return tail

def answer_yesno_prob(probb, q: str, fewshot_ctx: Optional[str]) -> str:
    prefix = SYS_PREFIX + (fewshot_ctx or "") + f"Pytanie: {q}\nOdpowiedz: "
    cands = ["tak", "nie"]
    scores = [
        conditional_logprob(probb["tok"], probb["model"], prefix, c, probb["device"])
        for c in cands
    ]
    return cands[0] if scores[0] >= scores[1] else cands[1]

def parse_binary_options(q: str) -> Optional[Tuple[str, str]]:
    qn = norm_text(q)
    m = re.search(r"(.+?)\s+czy\s+(.+?)\??$", qn)
    if not m:
        return None
    a = m.group(1).split()[-1]
    b = m.group(2).strip(" ?")
    return (a.strip(), b.strip())

def answer_binary_prob(probb, q: str, fewshot_ctx: Optional[str]) -> Optional[str]:
    opts = parse_binary_options(q)
    if not opts:
        return None
    prefix = SYS_PREFIX + (fewshot_ctx or "") + f"Pytanie: {q}\nOdpowiedz: "
    a, b = opts
    sa = conditional_logprob(probb["tok"], probb["model"], prefix, a, probb["device"])
    sb = conditional_logprob(probb["tok"], probb["model"], prefix, b, probb["device"])
    return a if sa >= sb else b

def load_dataset(q_path: str, a_path: str) -> List[Tuple[str, List[str]]]:
    with open(q_path, "r", encoding="utf-8") as fq, open(a_path, "r", encoding="utf-8") as fa:
        q_lines = [l.strip() for l in fq.readlines()]
        a_lines = [answers_from_line(l) for l in fa.readlines()]
    n = min(len(q_lines), len(a_lines))
    return [(q_lines[i], a_lines[i]) for i in range(n)]

def split_train_test(data: List[Tuple[str, List[str]]], train_ratio=0.8) -> Tuple[List, List]:
    idx = list(range(len(data)))
    random.shuffle(idx)
    cut = int(len(idx)*train_ratio)
    train = [data[i] for i in idx[:cut]]
    test = [data[i] for i in idx[cut:]]
    return train, test

def build_grouped_fewshot(train: List[Tuple[str, List[str]]]) -> Dict[str, List[Tuple[str,str]]]:
    groups = {"yesno": [], "binary": [], "math": [], "other": []}
    for q, golds in train:
        if is_math_like(q):
            groups["math"].append((q, golds[0]))
        elif is_yesno(q):
            groups["yesno"].append((q, golds[0]))
        elif is_binary_choice(q):
            groups["binary"].append((q, golds[0]))
        else:
            groups["other"].append((q, golds[0]))
    return groups

def pick_fewshot(groups: Dict[str, List[Tuple[str,str]]], gname: str, k=6) -> str:
    pool = groups.get(gname, [])
    if not pool:
        pool = sum(groups.values(), [])
    return build_fewshot(pool, k=k) if pool else ""

def main():
    set_seed(RNG_SEED)

    data = load_dataset(DEF_Q_PATH, DEF_A_PATH)
    print(f"[DATA] {len(data)} par Q/A")
    train, test = split_train_test(data, train_ratio=0.8)
    print(f"[SPLIT] train={len(train)} test={len(test)}")

    groups_train = build_grouped_fewshot(train)
    fewshot_ctx = {g: pick_fewshot(groups_train, g, k=6) for g in ["yesno","binary","math","other"]}

    genb  = load_generation_model()
    probb = load_probability_model()

    def group_of(q: str):
        if is_math_like(q): return "math"
        if is_binary_choice(q): return "binary"
        if is_yesno(q): return "yesno"
        return "other"

    found = []
    corr_lines = []

    iterator = tqdm(test, total=len(test), desc="Odpowiadanie", unit="pyt")

    for q, golds in iterator:
        grp = group_of(q)
        pred = None
        if grp == "math":
            pred = solve_math(q)
        elif grp == "binary":
            pred = answer_binary_prob(probb, q, fewshot_ctx.get("binary"))
        elif grp == "yesno":
            pred = answer_yesno_prob(probb, q, fewshot_ctx.get("yesno"))
        else:
            pred = gen_answer(genb, q, fewshot_ctx.get("other"))

        pred = (pred or "").strip()
        found.append(pred)
        corr_lines.append("\t".join(golds))

    with open(OUT_FOUND, "w", encoding="utf-8") as f:
        for a in found:
            f.write(a.lower().strip() + "\n")
    with open(OUT_CORR, "w", encoding="utf-8") as f:
        for c in corr_lines:
            f.write(c.lower().strip() + "\n")

if __name__ == "__main__":
    main()


Writing qa_polka_compat.py


In [ ]:
!python qa_polka_compat.py

2025-11-06 15:12:17.926223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762441937.946250    1489 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762441937.952343    1489 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762441937.967379    1489 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762441937.967406    1489 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762441937.967410    1489 computation_placer.cc:177] computation placer alr